# CZ validation

This notebook compares the Python PPS surrogate against PennyLane exact expectation values on small circuits (<=5 qubits) that contain multiple `CZ` gates.

In [ ]:
from pathlib import Path
import contextlib
import io
import os
import sys

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

os.environ.setdefault("MPLCONFIGDIR", str((root / ".mplconfig").resolve()))
(root / ".mplconfig").mkdir(exist_ok=True)

import numpy as np
import torch

from src.pauli_surrogate_python import (
    CliffordGate,
    PauliRotation,
    PauliSum,
    apply_clifford,
    evaluate_surrogate,
    make_pauli_string,
    propagate_surrogate,
    zero_filter,
)
from src_tensor.api import pennylane_expvals_small

torch.set_default_dtype(torch.float64)


In [ ]:
def build_observable(n_qubits, terms):
    obs = PauliSum(n_qubits)
    for pauli, coeff, qubits in terms:
        obs.add_from_str(pauli, coeff, qubits=qubits)
    return obs


def compile_python_pps(circuit, observables):
    compiled = []
    for obs in observables:
        with contextlib.redirect_stdout(io.StringIO()):
            surrogate = propagate_surrogate(circuit, obs)
            surrogate = zero_filter(surrogate)
        compiled.append(surrogate)
    return compiled


def evaluate_python_pps(compiled, thetas):
    return np.asarray([evaluate_surrogate(s, thetas) for s in compiled], dtype=np.float64)


local = ["I", "X", "Y", "Z"]
for control in local:
    for target in local:
        pstr = make_pauli_string(control + target)
        out_cz, sign_cz = apply_clifford(CliffordGate("CZ", [0, 1]), pstr)

        seq = pstr
        sign_seq = 1
        for gate in (
            CliffordGate("H", [1]),
            CliffordGate("CNOT", [0, 1]),
            CliffordGate("H", [1]),
        ):
            seq, step_sign = apply_clifford(gate, seq)
            sign_seq *= step_sign

        assert (out_cz, sign_cz) == (seq, sign_seq)

print("Local CZ conjugation agrees with H(target) -> CNOT -> H(target) on all 16 two-qubit Paulis.")


In [ ]:
def case_q3():
    n_qubits = 3
    circuit = [
        CliffordGate("H", [0]),
        CliffordGate("H", [1]),
        PauliRotation("X", [0], param_idx=0),
        PauliRotation("Y", [1], param_idx=1),
        CliffordGate("CZ", [0, 1]),
        CliffordGate("CZ", [1, 2]),
        PauliRotation("ZZ", [0, 2], param_idx=2),
        CliffordGate("CZ", [0, 2]),
        PauliRotation("X", [2], param_idx=3),
    ]
    observables = [
        build_observable(n_qubits, [("X", 0.7, [0]), ("YZ", -0.25, [1, 2])]),
        build_observable(n_qubits, [("ZX", 0.5, [0, 2]), ("Y", -0.3, [1])]),
        build_observable(n_qubits, [("XX", -0.4, [0, 1]), ("Z", 0.15, [2])]),
    ]
    thetas = np.asarray([0.31, -0.47, 0.22, 0.15], dtype=np.float64)
    return {
        "name": "q3_multi_cz",
        "n_qubits": n_qubits,
        "circuit": circuit,
        "observables": observables,
        "thetas": thetas,
    }


def case_q4():
    n_qubits = 4
    circuit = [
        CliffordGate("H", [0]),
        CliffordGate("H", [2]),
        PauliRotation("Y", [0], param_idx=0),
        PauliRotation("X", [3], param_idx=1),
        CliffordGate("CZ", [0, 1]),
        CliffordGate("CZ", [1, 2]),
        PauliRotation("XX", [1, 2], param_idx=2),
        CliffordGate("CZ", [2, 3]),
        PauliRotation("Z", [2], param_idx=3),
        CliffordGate("CZ", [0, 3]),
        CliffordGate("CZ", [1, 3]),
    ]
    observables = [
        build_observable(n_qubits, [("Y", 0.4, [0]), ("XZ", -0.2, [1, 3])]),
        build_observable(n_qubits, [("XY", 0.35, [0, 2]), ("ZZ", 0.5, [1, 3])]),
        build_observable(n_qubits, [("X", -0.6, [3]), ("YZZ", 0.1, [0, 1, 2])]),
    ]
    thetas = np.asarray([0.12, -0.37, 0.44, -0.28], dtype=np.float64)
    return {
        "name": "q4_multi_cz",
        "n_qubits": n_qubits,
        "circuit": circuit,
        "observables": observables,
        "thetas": thetas,
    }


def case_q5():
    n_qubits = 5
    circuit = [
        CliffordGate("H", [0]),
        CliffordGate("H", [2]),
        CliffordGate("H", [4]),
        PauliRotation("X", [1], param_idx=0),
        PauliRotation("Y", [3], param_idx=1),
        CliffordGate("CZ", [0, 1]),
        CliffordGate("CZ", [1, 2]),
        CliffordGate("CZ", [2, 3]),
        CliffordGate("CZ", [3, 4]),
        PauliRotation("ZX", [0, 2], param_idx=2),
        CliffordGate("CZ", [0, 4]),
        PauliRotation("YZ", [1, 4], param_idx=3),
        CliffordGate("CZ", [1, 3]),
        PauliRotation("XXX", [0, 2, 4], param_idx=4),
        CliffordGate("CZ", [0, 2]),
        CliffordGate("CZ", [2, 4]),
    ]
    observables = [
        build_observable(n_qubits, [("X", 0.25, [0]), ("Y", -0.45, [4]), ("ZZ", 0.2, [1, 3])]),
        build_observable(n_qubits, [("XY", 0.3, [1, 2]), ("ZZX", -0.15, [0, 3, 4])]),
        build_observable(n_qubits, [("YXZ", 0.2, [0, 2, 4]), ("Z", 0.35, [3])]),
    ]
    thetas = np.asarray([0.18, -0.52, 0.41, -0.33, 0.27], dtype=np.float64)
    return {
        "name": "q5_multi_cz",
        "n_qubits": n_qubits,
        "circuit": circuit,
        "observables": observables,
        "thetas": thetas,
    }


cases = [case_q3(), case_q4(), case_q5()]


In [ ]:
results = []

for case in cases:
    compiled = compile_python_pps(case["circuit"], case["observables"])
    pps_vals = evaluate_python_pps(compiled, case["thetas"])
    exact_vals = pennylane_expvals_small(
        circuit=case["circuit"],
        observables=case["observables"],
        thetas=torch.tensor(case["thetas"]),
        n_qubits=case["n_qubits"],
        max_qubits=5,
    ).detach().cpu().numpy()
    max_abs_err = float(np.max(np.abs(pps_vals - exact_vals)))
    n_cz = sum(1 for gate in case["circuit"] if isinstance(gate, CliffordGate) and gate.symbol == "CZ")

    results.append(
        {
            "case": case["name"],
            "n_qubits": case["n_qubits"],
            "n_cz": n_cz,
            "pps": pps_vals,
            "exact": exact_vals,
            "max_abs_err": max_abs_err,
        }
    )

    print(f"{case['name']} | qubits={case['n_qubits']} | cz={n_cz}")
    print("  pps  :", np.array2string(pps_vals, precision=12))
    print("  exact:", np.array2string(exact_vals, precision=12))
    print(f"  max_abs_err: {max_abs_err:.3e}")
    print()

overall_max_err = max(item["max_abs_err"] for item in results)
print(f"overall max_abs_err = {overall_max_err:.3e}")
assert overall_max_err < 1e-9
results
